# Flow matching: watching probability mass move

**Presentation goal.** Show one time-dependent velocity field and follow particles as it transports a simple base distribution into a bimodal target distribution.

This notebook recreates the central visual idea from Peter Roelants' *Flow Matching: A visual introduction*: horizontal position is flow time $t$, vertical position is the state $x_t$, and color shows the signed velocity $v_t(x)$. The implementation here is independent and deliberately small.

> **Take-away:** a flow-matching model does not memorize trajectories. It learns a vector field. New samples are generated by drawing $x_0$ from noise and solving $\frac{dx_t}{dt}=v_t(x_t)$.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import TwoSlopeNorm
import numpy as np
from IPython.display import Image, display
from scipy.stats import ks_2samp, norm

SEED = 7
rng = np.random.default_rng(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})

# Work both from the repository root and from this notebook's directory.
candidate = Path("research_notes/seminar_presentation/practical_example")
OUTPUT_DIR = candidate if candidate.exists() else Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HERO_FILE = OUTPUT_DIR / "00_flow_matching_vector_field.png"
ANIMATION_FILE = OUTPUT_DIR / "00_flow_matching_particles.gif"

print(f"Figures will be written to: {OUTPUT_DIR.resolve()}")

## 1. Source and target distributions

We transport standard Gaussian noise

$$X_0 \sim \mathcal N(0,1)$$

into a two-component Gaussian mixture

$$X_1 \sim 0.55\,\mathcal N(-0.85,0.65^2) + 0.45\,\mathcal N(1.50,0.25^2).$$

These parameters follow the toy example in Roelants (2025), making the resulting figure easy to compare with the linked article.

In [ ]:
WEIGHTS = np.array([0.55, 0.45])
MEANS = np.array([-0.85, 1.50])
SIGMAS = np.array([0.65, 0.25])


def target_pdf(x):
    """Density of the bimodal target distribution."""
    x = np.asarray(x)
    return np.sum(
        WEIGHTS * norm.pdf(x[..., None], loc=MEANS, scale=SIGMAS),
        axis=-1,
    )


def sample_target(size, generator=rng):
    """Draw samples from the bimodal target distribution."""
    component = generator.choice(len(WEIGHTS), size=size, p=WEIGHTS)
    return generator.normal(MEANS[component], SIGMAS[component])


x_plot = np.linspace(-3.2, 3.2, 600)
fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
ax.plot(x_plot, norm.pdf(x_plot), color="#F28E2B", lw=2.5, label=r"base $p_0$: Gaussian noise")
ax.fill_between(x_plot, norm.pdf(x_plot), color="#F28E2B", alpha=0.18)
ax.plot(x_plot, target_pdf(x_plot), color="#4E79A7", lw=2.5, label=r"target $p_1$: bimodal data")
ax.fill_between(x_plot, target_pdf(x_plot), color="#4E79A7", alpha=0.18)
ax.set(xlabel="state x", ylabel="density", title="Where the particles start — and where they should end")
ax.legend(frameon=False, ncol=2)
plt.show()

## 2. The vector field

Training pairs use independent draws $(X_0,X_1)$ and the straight conditional path

$$X_t=(1-t)X_0+tX_1, \qquad U_t=X_1-X_0.$$

A squared-error flow-matching model learns the conditional mean

$$v_t^*(x)=\mathbb E[X_1-X_0\mid X_t=x].$$

For this Gaussian-mixture toy problem, that population-optimal field is available analytically. The code below evaluates it directly. Thus the notebook focuses on **what the learned field does**, without introducing approximation error or a neural-network training loop. In a realistic problem, a neural network approximates the same conditional expectation from samples.

In [ ]:
def velocity_field(t, x):
    """Population-optimal linear flow-matching field E[X1-X0 | Xt=x].

    X0 ~ N(0,1), X1 is the Gaussian mixture above, and
    Xt = (1-t) X0 + t X1. Inputs t and x may be broadcastable arrays.
    """
    t = np.asarray(t, dtype=float)
    x = np.asarray(x, dtype=float)
    te = t[..., None]
    xe = x[..., None]

    # Distribution of Xt conditional on target mixture component k.
    variance_t = (1.0 - te) ** 2 + te**2 * SIGMAS**2
    mean_t = te * MEANS
    component_density = WEIGHTS * norm.pdf(xe, loc=mean_t, scale=np.sqrt(variance_t))
    component_probability = component_density / component_density.sum(axis=-1, keepdims=True)

    # Gaussian conditioning within component k.
    covariance_velocity_xt = te * SIGMAS**2 - (1.0 - te)
    component_velocity = MEANS + (
        covariance_velocity_xt / variance_t * (xe - mean_t)
    )
    return np.sum(component_probability * component_velocity, axis=-1)


def euler_paths(x0, n_steps=120):
    """Integrate dx/dt = v_t(x) with explicit Euler steps."""
    times = np.linspace(0.0, 1.0, n_steps + 1)
    paths = np.empty((n_steps + 1, len(np.atleast_1d(x0))))
    paths[0] = np.atleast_1d(x0)
    for i, (t_left, t_right) in enumerate(zip(times[:-1], times[1:])):
        dt = t_right - t_left
        paths[i + 1] = paths[i] + dt * velocity_field(t_left, paths[i])
    return times, paths


# Evaluate the field on a grid and integrate a few representative particles.
t_grid = np.linspace(0.0, 1.0, 181)
x_grid = np.linspace(-3.2, 3.2, 321)
T, X = np.meshgrid(t_grid, x_grid, indexing="xy")
V = velocity_field(T, X)

x0_highlight = norm.ppf(np.linspace(0.04, 0.96, 13))
times, highlighted_paths = euler_paths(x0_highlight)

## 3. One field, many trajectories

- **Red:** positive velocity, so particles move upward toward larger $x$.
- **Blue:** negative velocity, so particles move downward toward smaller $x$.
- **White:** velocity near zero.
- Curves are not training pairs; they are solutions of the single marginal ODE field.

In [ ]:
fig, (ax_left, ax_field, ax_right) = plt.subplots(
    1, 3, figsize=(13.5, 6.3), sharey=True,
    gridspec_kw={"width_ratios": [1.2, 5.5, 1.2], "wspace": 0.04},
)

# Symmetric normalization makes color directly encode signed velocity.
limit = np.quantile(np.abs(V), 0.98)
color_norm = TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)
field_image = ax_field.pcolormesh(T, X, V, cmap="coolwarm", norm=color_norm, shading="auto", rasterized=True)

# A sparse set of arrows emphasizes that this is a vector field in (t, x).
t_arrow = np.linspace(0.04, 0.96, 13)
x_arrow = np.linspace(-2.8, 2.8, 15)
TA, XA = np.meshgrid(t_arrow, x_arrow)
VA = velocity_field(TA, XA)
speed = np.sqrt(1.0 + VA**2)
ax_field.quiver(
    TA, XA, 1.0 / speed, VA / speed,
    color="black", alpha=0.32, scale=30, width=0.0022,
    headwidth=3.2, headlength=4.0, pivot="mid",
)

path_colors = plt.cm.viridis(np.linspace(0.05, 0.95, highlighted_paths.shape[1]))
for path, color in zip(highlighted_paths.T, path_colors):
    ax_field.plot(times, path, color=color, lw=2.0, alpha=0.95)
ax_field.scatter(np.zeros_like(x0_highlight), x0_highlight, c=path_colors, s=28, edgecolor="white", lw=0.5, zorder=5)
ax_field.scatter(np.ones_like(x0_highlight), highlighted_paths[-1], c=path_colors, s=28, edgecolor="white", lw=0.5, zorder=5)
ax_field.set(xlim=(0, 1), ylim=(-3.2, 3.2), xlabel=r"flow time $t$", ylabel=r"state $x_t$",
             title="The velocity field bends Gaussian noise into two modes")
ax_field.grid(False)

# Marginal densities bookend the field.
base_density = norm.pdf(x_grid)
final_density = target_pdf(x_grid)
ax_left.plot(base_density, x_grid, color="#F28E2B", lw=2.5)
ax_left.fill_betweenx(x_grid, 0, base_density, color="#F28E2B", alpha=0.28)
ax_left.invert_xaxis()
ax_left.set(title=r"base $p_0$", xlabel="density", ylabel=r"state $x_t$")

ax_right.plot(final_density, x_grid, color="#4E79A7", lw=2.5)
ax_right.fill_betweenx(x_grid, 0, final_density, color="#4E79A7", alpha=0.28)
ax_right.set(title=r"target $p_1$", xlabel="density")
ax_right.yaxis.set_label_position("right")
ax_right.tick_params(labelleft=False, labelright=True)

cbar = fig.colorbar(field_image, ax=ax_field, orientation="horizontal", pad=0.13, fraction=0.055, aspect=35)
cbar.set_label(r"signed velocity $v_t(x)$   (blue: down, red: up)")
fig.suptitle("Flow matching generates samples by integrating one learned vector field", fontsize=16, fontweight="bold", y=1.01)
fig.savefig(HERO_FILE, dpi=180, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved presentation figure: {HERO_FILE.resolve()}")

## 4. Animate particles flowing through the field

Each dot starts as an independent draw from $p_0$. Its trail is obtained by Euler integration. Particle color is fixed by its starting rank so individual paths remain easy to follow.

In [ ]:
animation_rng = np.random.default_rng(SEED + 1)
x0_animation = np.sort(animation_rng.normal(size=70))
animation_times, animation_paths = euler_paths(x0_animation, n_steps=80)
particle_colors = plt.cm.turbo(np.linspace(0.03, 0.97, len(x0_animation)))

fig, ax = plt.subplots(figsize=(10.5, 6.0))
ax.pcolormesh(T, X, V, cmap="coolwarm", norm=color_norm, shading="auto", alpha=0.88)
ax.set(xlim=(0, 1), ylim=(-3.2, 3.2), xlabel=r"flow time $t$", ylabel=r"state $x_t$")
ax.grid(False)
title = ax.set_title("", fontsize=14, fontweight="bold")
trail_lines = [ax.plot([], [], color=c, lw=0.8, alpha=0.48)[0] for c in particle_colors]
particles = ax.scatter(
    np.zeros(len(x0_animation)), x0_animation,
    c=particle_colors, s=27, edgecolor="white", linewidth=0.35, zorder=5,
)


def update(frame):
    for j, line in enumerate(trail_lines):
        line.set_data(animation_times[: frame + 1], animation_paths[: frame + 1, j])
    particles.set_offsets(np.column_stack([
        np.full(len(x0_animation), animation_times[frame]),
        animation_paths[frame],
    ]))
    title.set_text(f"Integrating the vector field:  t = {animation_times[frame]:.2f}")
    return [*trail_lines, particles, title]


animation = FuncAnimation(fig, update, frames=len(animation_times), interval=70, blit=True)
animation.save(ANIMATION_FILE, writer=PillowWriter(fps=14), dpi=105)
plt.close(fig)

print(f"Saved presentation animation: {ANIMATION_FILE.resolve()}")
display(Image(filename=str(ANIMATION_FILE)))

## 5. Small quantitative check

A compelling animation is not evidence by itself. We therefore transport a larger base sample and compare its endpoint with fresh ground-truth target draws. The overlay and two-sample Kolmogorov–Smirnov statistic check the full one-dimensional distribution, not just a few paths.

In [ ]:
check_rng = np.random.default_rng(SEED + 2)
n_check = 10_000
x0_check = check_rng.normal(size=n_check)
target_check = sample_target(n_check, generator=check_rng)
_, check_paths = euler_paths(x0_check, n_steps=240)
generated = check_paths[-1]
ks = ks_2samp(generated, target_check)

fig, ax = plt.subplots(figsize=(9, 3.5), constrained_layout=True)
bins = np.linspace(-3.2, 3.2, 80)
ax.hist(generated, bins=bins, density=True, alpha=0.48, color="#59A14F", label="transported samples")
ax.plot(x_plot, target_pdf(x_plot), color="#4E79A7", lw=2.5, label="target density")
ax.set(xlabel="state x", ylabel="density", title="Endpoint check: transported samples reproduce the bimodal target")
ax.legend(frameon=False)
plt.show()

print(f"Generated mean / target mean: {generated.mean(): .3f} / {target_check.mean(): .3f}")
print(f"Generated std  / target std:  {generated.std(): .3f} / {target_check.std(): .3f}")
print(f"Two-sample KS statistic:       {ks.statistic: .4f}  (p={ks.pvalue:.3f})")
assert ks.statistic < 0.03, "Endpoint mismatch is unexpectedly large; check the field or integrator."
print("Check passed: KS statistic < 0.03.")

## What to say on the slide

1. “The background is the learned object: a velocity for every time and state.”
2. “Sampling starts with Gaussian noise and numerically integrates this field.”
3. “A single smooth field splits probability mass into two modes; no trajectory was stored.”
4. “The curved paths arise even though the conditional training paths were straight, because the marginal field averages over many possible pairings.”

### Scope and limitation

This is an unconditional 1D population example. It demonstrates the mechanics of flow matching, not neural posterior estimation. In FMPE, the state becomes the parameter vector $\theta$ and the velocity field is additionally conditioned on an observation $x_{\mathrm{obs}}$.

## References and attribution

- Roelants, P. (2025). [*Flow Matching: A visual introduction*](https://peterroelants.github.io/posts/flow_matching_intro/). The toy-distribution parameters and the position-versus-time visualization in this notebook are adapted from this article. Accessed 17 July 2026.
- Roelants, P. (2025). [Source notebook: `flow_matching_intro.ipynb`](https://github.com/peterroelants/peterroelants.github.io/blob/main/notebooks/diffusion_flow_matching/flow_matching_intro.ipynb). The implementation in the present notebook was written independently and replaces neural training with the analytic population field.
- Lipman, Y., Chen, R. T. Q., Ben-Hamu, H., Nickel, M., & Le, M. (2023). [*Flow Matching for Generative Modeling*](https://arxiv.org/abs/2210.02747). International Conference on Learning Representations (ICLR). Introduces flow matching as vector-field regression for continuous normalizing flows.
- Liu, X., Gong, C., & Liu, Q. (2023). [*Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow*](https://arxiv.org/abs/2209.03003). International Conference on Learning Representations (ICLR). Closely related straight-line interpolation objective.
- Holderrieth, P., Erives, E., & Leahy, J. (2024). [*Flow Matching Guide and Code*](https://arxiv.org/abs/2412.06264). A broader guide to probability paths, conditional flow matching, and implementation.

**Software:** NumPy, SciPy, Matplotlib, Pillow, and IPython. No code or generated media were copied from the source repository.